In [22]:
from ax.service.ax_client import AxClient
import sys
sys.path.append('../')
import helper_functions as hf


In [23]:
iteration_to_update = 0
optimizer_file_path = 'iteration_' + str(iteration_to_update) + '/optimizer/optimizer_'
ax_to_update_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"



In [24]:
ax_to_update = AxClient.load_from_json_file(ax_to_update_path)
trials_to_update = ax_to_update.get_trials_data_frame()
trials_to_update


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,surf_3_conc,drug_conc,surf_1,surf_2,surf_3
0,0,0_0,COMPLETED,GenerationStep_0,36.197205,0.2063,0.3073,0.0373,6.342538,19.792553,10.062115,25.0,s8,s6,s7
1,1,1_0,COMPLETED,GenerationStep_0,50.000000,0.2063,0.3073,0.0373,25.388954,2.403562,19.870581,25.0,s3,s2,s6
2,2,2_0,COMPLETED,GenerationStep_0,28.779619,0.4045,0.4196,0.0728,0.561928,27.771701,0.445989,25.0,s5,s7,s3
3,3,3_0,COMPLETED,GenerationStep_0,50.000000,0.4045,0.4196,0.0728,14.076332,3.422734,7.078612,25.0,s3,s3,s8
4,4,4_0,COMPLETED,GenerationStep_0,50.000000,0.2962,0.4364,0.0493,13.456925,31.466190,1.570027,25.0,s1,s4,s5
5,5,5_0,COMPLETED,GenerationStep_0,50.000000,0.2962,0.4364,0.0493,20.019197,14.549110,11.283737,25.0,s6,s8,s7
6,6,6_0,COMPLETED,GenerationStep_0,50.000000,0.3528,0.2810,0.0711,2.750314,12.241791,9.081532,25.0,s1,s1,s6
7,7,7_0,COMPLETED,GenerationStep_0,50.000000,0.3528,0.2810,0.0711,5.021025,0.862688,17.897261,25.0,s6,s5,s1


In [25]:
def update_data_to_optimizer(ax_client, list_of_new_failures, list_of_new_success):

    # Load the existing optimizer state
    before_update_path = optimizer_file_path + f"{iteration_to_update}_before_update.json"
    updated_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"

    ax_client.save_to_json_file(before_update_path)

    # Fetch current trials
    trials_df = ax_client.get_trials_data_frame()

    if len(list_of_new_failures) != 0:
        for trial_index in list_of_new_failures:
            # Make sure we actually have this trial
            if trial_index not in trials_df["trial_index"].values:
                print(f"Trial {trial_index} not found – skipping.")
                continue

            # Build the forced-failure payload
            new_failure = {
                "obj_surf_conc": hf.surfactant_stock_conc,
            }
            # Update the trial in-place
            ax_client.update_trial_data(trial_index=trial_index, raw_data=new_failure)
            print(f"Updated trial {trial_index}: set obj_surf_conc = {hf.surfactant_stock_conc}")

    if len(list_of_new_success) != 0:
        for trial_index in list_of_new_success:
            # Make sure we actually have this trial
            if trial_index not in trials_df["trial_index"].values:
                print(f"Trial {trial_index} not found – skipping.")
                continue    

            surf_conc_success = trials_df[trials_df['trial_index'] == trial_index]['surf_1_conc'].values[0] + trials_df[trials_df['trial_index'] == trial_index]['surf_2_conc'].values[0] + trials_df[trials_df['trial_index'] == trial_index]['surf_3_conc'].values[0]
            new_success = {
                "obj_surf_conc": surf_conc_success,
            }

            # Update the trial in-place
            ax_client.update_trial_data(trial_index=trial_index, raw_data=new_success)
            print(f"Updated trial {trial_index}: set obj_surf_conc = {surf_conc_success}")

    ax_client.save_to_json_file(updated_path)
    return ax_client

In [26]:
list_of_new_failures = []


list_of_new_success = []

In [27]:
print("Please double check the results you are updating before continue...")
print("*" * 100)
print("*" * 100)
print("Changing them from SUCCESS to FAILURE")

df = trials_to_update[trials_to_update["trial_index"].isin(list_of_new_failures)]
df


Please double check the results you are updating before continue...
****************************************************************************************************
****************************************************************************************************
Changing them from SUCCESS to FAILURE


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,surf_3_conc,drug_conc,surf_1,surf_2,surf_3


In [28]:
print("Please double check the results you are updating before continue...")
print("*" * 100)
print("*" * 100)
print("Changing them from FAILURE to SUCCESS")

df = trials_to_update[trials_to_update["trial_index"].isin(list_of_new_success)]
df

Please double check the results you are updating before continue...
****************************************************************************************************
****************************************************************************************************
Changing them from FAILURE to SUCCESS


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,surf_3_conc,drug_conc,surf_1,surf_2,surf_3


In [29]:
new_ax_client = update_data_to_optimizer(ax_client = ax_to_update, list_of_new_failures = list_of_new_failures, list_of_new_success = list_of_new_success)


In [30]:
updated_trials = new_ax_client.get_trials_data_frame()
updated_trials[updated_trials["trial_index"].isin(list_of_new_failures + list_of_new_success)]

,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,surf_3_conc,drug_conc,surf_1,surf_2,surf_3
